# Multilevel and Marginal Modeling in Python

---

## 1. Introduction
This notebook builds on our earlier introduction to regression modeling with NHANES data. Here, we extend those basic [linear](../02_fitting_models_to_independent_data/09_linear_regression_in_python.ipynb) and [logistic](../02_fitting_models_to_independent_data/10_logistic_regression_in_python.ipynb) regression methods to more advanced approaches for analyzing data with statistical dependencies.

Some of the models in this notebook may take a few minutes to run.

Let's start by importing the necessary libraries.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

Data often has a multilevel or dependent structure for many reasons. In fact, most real-world datasets show some form of dependence, so independent data should be seen as the exception. In this notebook, we’ll revisit the NHANES data with a focus on dependence caused by clustering, which we’ll define shortly.

We start by reading the data from a CSV file into a Pandas dataframe and remove rows with missing values in the variables we need. (While there are better ways to handle missing data, we use this simple approach for now.)

In addition to the demographic and health variables used previously, we keep two extra variables—SDMVSTRA and SDMVPSU—that will help us define the clustering structure in the data.

In [ ]:
# Read the data file
nhanes_df = pd.read_csv("../02_fitting_models_to_independent_data/data/nhanes_2015_2016.csv")

# Drop unused columns, drop rows with any missing values
cols = ["BPXSY1", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "BMXBMI", "SMQ020", "SDMVSTRA", "SDMVPSU"]
nhanes_df = nhanes_df[cols].dropna()

# Rename columns for convenience
nhanes_df = nhanes_df.rename(columns={
    "BPXSY1": "sbp", 
    "RIDAGEYR": "age", 
    "RIAGENDR": "gender", 
    "RIDRETH1": "ethnicity", 
    "DMDEDUC2": "education",
    "BMXBMI": "bmi", 
    "SMQ020": "smoked", 
    "SDMVSTRA": "stratum", 
    "SDMVPSU": "psu",
})

# Create the labeled versions of `gender` and `education` predictors
nhanes_df["gender_labeled"] = nhanes_df["gender"].replace({1: "Male", 2:"Female"})
nhanes_df["education_labeled"] = nhanes_df["education"].replace({
    1: "lt9",
    2: "xp_11",
    3: "HS",
    4: "SomeCollege",
    5: "College",
    7: np.nan, 
    9: np.nan
})

# Simplify the answers for the `smoked` predictor
nhanes_df["smoked"] = nhanes_df["smoked"].replace({2: 0, 7: np.nan, 9: np.nan})

# Display the first few rows
nhanes_df.head()

,sbp,age,gender,ethnicity,education,bmi,smoked,stratum,psu,gender_labeled,education_labeled
0,128.0,62,1,3,5.0,27.8,1.0,125,1,Male,College
1,146.0,53,1,3,3.0,30.8,1.0,125,1,Male,HS
2,138.0,78,1,3,3.0,28.8,1.0,131,1,Male,HS
3,132.0,56,2,3,5.0,42.4,0.0,131,1,Female,College
4,100.0,42,2,4,4.0,20.3,0.0,126,2,Female,SomeCollege



### Introduction to Clustered Data

Data are often dependent because they are collected using "cluster sampling." In this approach, the population is divided into groups (clusters), a few clusters are selected, and then individuals are sampled from those clusters. This is how NHANES collects its data. Instead of sampling people from all over the country, NHANES sets up examination centers in selected communities and examines many people at each center.

Cluster sampling isn’t the only reason for dependence in data. For example, in longitudinal studies, the same people are measured multiple times, so their measurements are naturally correlated. However, since NHANES uses cluster sampling and isn’t longitudinal, we’ll focus on clustering as our example of dependent data.

In any cluster sample, people within the same cluster tend to be more similar to each other than to people in other clusters. For NHANES, clusters are based on geography, so people from the same community may share similar characteristics. It’s important to account for this dependence in our analysis.

### Clustering Structure in NHANES

The NHANES sampling process is complex, but we’ll keep it simple here. (More details are available [here](https://wwwn.cdc.gov/nchs/nhanes/analyticguidelines.aspx), but you don’t need them for this course.) In short, NHANES selects a sample of US counties, then subregions within those counties, and finally people within those subregions. Because people from the same county live near each other, they are likely to be more similar than people from different counties.

For privacy reasons, NHANES does not provide the actual county identifiers. Instead, we get "masked variance units" (MVUs), which are artificial groups created by combining subregions from different counties. These MVUs aren’t real geographic clusters, but they are designed to mimic them while protecting participants’ privacy.

In this notebook, we’ll treat MVUs as our clusters and examine how clustering affects some NHANES variables.

We can identify each MVU by combining the `stratum` and `psu` variables:

In [9]:
nhanes_df["group"] = 10 * nhanes_df["stratum"] + nhanes_df["psu"]
nhanes_df.head()

,sbp,age,gender,ethnicity,education,bmi,smoked,stratum,psu,gender_labeled,education_labeled,group
0,128.0,62,1,3,5.0,27.8,1.0,125,1,Male,College,1251
1,146.0,53,1,3,3.0,30.8,1.0,125,1,Male,HS,1251
2,138.0,78,1,3,3.0,28.8,1.0,131,1,Male,HS,1311
3,132.0,56,2,3,5.0,42.4,0.0,131,1,Female,College,1311
4,100.0,42,2,4,4.0,20.3,0.0,126,2,Female,SomeCollege,1262



---

## 2. Data Understanding

### Intraclass Correlation
The **intraclass correlation coefficient (ICC)** measures how similar observations are within the same cluster. Unlike Pearson's correlation, the ICC ranges from 0 (no similarity within clusters) to 1 (everyone in a cluster is identical).

We can estimate the ICC using two methods: *marginal regression* and *multilevel regression*. We'll start with **Generalized Estimating Equations (GEE)** to fit marginal models and estimate the ICC for NHANES clusters.

Let's first look at the ICC for systolic bloop pressure `sbp`:

In [10]:
model = sm.GEE.from_formula("sbp ~ 1", groups="group", cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.030


The estimated ICC is 0.03. This is small, but not negligible. Although the ICC is a kind of correlation, its values aren't directly comparable to Pearson correlation coefficitents. For example, 0.03 would be very small for Person's $r$, but is not unusually small for an ICC.

To better understand clustering in the data, we also calculate ICC values for several other variables used in our analyses.

In [12]:
for col in ["sbp", "age", "bmi", "smoked", "stratum"]:
    model = sm.GEE.from_formula(f"{col} ~ 1", groups="group", cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
    result = model.fit()
    print(f"{col}:", result.cov_struct.summary())

sbp: The correlation between two observations in the same cluster is 0.030
age: The correlation between two observations in the same cluster is 0.035
bmi: The correlation between two observations in the same cluster is 0.039
smoked: The correlation between two observations in the same cluster is 0.026
stratum: The correlation between two observations in the same cluster is 0.959


The values are generally similar to what we saw for blood pressure, except for `stratum`, which is one component of the cluster definition itself, and therefore has a very high ICC.

To illustrate that the ICC values shown above are not consistent with a complete absence of dependence, we simulate 10 sets of random data and calculate the ICC value for each set:

In [18]:
# Set a random seed for reproducibility
np.random.seed(42)

for k in range(10):
    nhanes_df["noise"] = np.random.normal(size=nhanes_df.shape[0])
    model = sm.GEE.from_formula("noise ~ 1", groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
    result = model.fit()
    print(result.cov_struct.summary())

The correlation between two observations in the same cluster is -0.001
The correlation between two observations in the same cluster is -0.002
The correlation between two observations in the same cluster is -0.002
The correlation between two observations in the same cluster is 0.001
The correlation between two observations in the same cluster is -0.000
The correlation between two observations in the same cluster is -0.002
The correlation between two observations in the same cluster is -0.001
The correlation between two observations in the same cluster is 0.000
The correlation between two observations in the same cluster is 0.003
The correlation between two observations in the same cluster is -0.000


We see that the estimated ICC for pure simulated noise is random but highly concentrated near zero, varying from around `-0.002` to `+0.003`.  These values are at least a factor of 10 smaller than the ICC values obtaine with the actual NHANES data.  Thus, while the ICC values for the NHANES data are numerically small, they are much larger than what we would expect to obtain if the observations were independent.

### Conditional Intraclass Correlation
The ICCs we looked at before were marginal -they measured how similar values (like SBP) are within clusters compared to between clusters. However, these "cluster effects" might just reflect demographic differences among clusters. For exampla, clusters with older people may have higher SBP overall.

If we control for age, the ICC might get smaller, since we've accounted for a major source of variation. We'll see this effect in the next analysis.

In [23]:
model = sm.GEE.from_formula("sbp ~ age", groups="group", cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.019


The ICC for SBP drops from 0.03 to 0.02.  We can now assess whether it
drops even further when we add additional covariates that we know to
be predictive of blood pressure.

In [29]:
model = sm.GEE.from_formula("sbp ~ age + gender_labeled + bmi + C(ethnicity)",
           groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=nhanes_df)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.013


We see here that the ICC has further reduced, to 0.013, due to controlling for these additional factors including ethnicity.

> **Note:** The variable `ethnicity` is a categorical variable containing 5 levels of race/ethnicity information. Since NHANES categorical variables are coded numerically, Statsmodels would have no way of knowing that these are codes and not quantitative data, thus we must use the C() syntax in the formula above to force this variable to be treated as being categorical. 

---

## 3. Fitting Marginal Linear Models